# 05 — F10.7 and CMAM: Common-Period Comparison

This notebook aligns daily F10.7 and monthly CMAM temperature perturbations on a common monthly calendar, January 2005–June 2010. Daily F10.7 is aggregated to monthly means. Each series is transformed using monthly sampling and the same Morlet settings.

Because the signals have different units, normalized global spectra compare relative spectral shape rather than absolute power.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
F107_CSV = PROJECT_ROOT / "data" / "raw" / "f107" / "F107_daily_1979_2010.csv"
TEMP_CSV = PROJECT_ROOT / "data" / "processed" / "temperature_perturbation.csv"
sys.path.insert(0, str(PROJECT_ROOT / "wave_python"))
from waveletFunctions import wavelet

f = pd.read_csv(F107_CSV)
f.columns = [str(c).strip().lower() for c in f.columns]
f["date"] = pd.to_datetime(f["date"], errors="coerce")
f["f107"] = pd.to_numeric(f["f107"], errors="coerce")
f = f.dropna(subset=["date", "f107"]).sort_values("date").drop_duplicates("date")
f["month"] = f["date"].dt.to_period("M")
f_month = f.groupby("month", as_index=False)["f107"].mean()
f_month["date"] = f_month["month"].dt.to_timestamp()
f_month = f_month[["date", "f107"]]

t = pd.read_csv(TEMP_CSV)
t["date"] = pd.to_datetime(t["date"], errors="coerce")
temp_col = next(c for c in ["temperature_perturbation_K", "temperature_perturbation"] if c in t.columns)
t[temp_col] = pd.to_numeric(t[temp_col], errors="coerce")
t = t.dropna(subset=["date", temp_col]).sort_values("date")
t["date"] = t["date"].dt.to_period("M").dt.to_timestamp()
t = t[["date", temp_col]].rename(columns={temp_col: "temperature_perturbation"})
merged = f_month.merge(t, on="date", how="inner").sort_values("date")
merged = merged[(merged["date"] >= "2005-01-01") & (merged["date"] <= "2010-06-01")]
if len(merged) < 24:
    raise ValueError(f"Only {len(merged)} common months found.")
print("Common months:", len(merged), "range:", merged.date.min(), "to", merged.date.max())
display(merged.head())


In [ ]:
dt = 365.2425 / 12
pad, dj, s0 = 1, 0.25, 2 * dt
mother, param = "MORLET", 6
j1 = int(np.floor(np.log2(len(merged) * dt / s0) / dj))

solar = merged["f107"].to_numpy(float)  # no mean subtraction
temp = merged["temperature_perturbation"].to_numpy(float)  # as prepared
wave_s, period_s, _, _ = wavelet(solar, dt, pad, dj, s0, j1, mother, param)
wave_t, period_t, _, _ = wavelet(temp, dt, pad, dj, s0, j1, mother, param)
if not np.allclose(period_s, period_t):
    raise ValueError("Period grids do not match.")
power_s = np.abs(wave_s) ** 2
power_t = np.abs(wave_t) ** 2
global_s = power_s.mean(axis=1)
global_t = power_t.mean(axis=1)

mask = (period_s >= 60) & (period_s <= 730)
norm_s = global_s[mask] / global_s[mask].max()
norm_t = global_t[mask] / global_t[mask].max()
period_display = period_s[mask]

print("Solar dominant period in display range (days):", period_display[np.argmax(global_s[mask])])
print("Temperature dominant period in display range (days):", period_display[np.argmax(global_t[mask])])


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(period_display, norm_s, marker="o", label="F10.7 monthly mean")
ax.plot(period_display, norm_t, marker="o", label="CMAM temperature perturbation")
ax.axvline(365, color="gray", linestyle=":", label="365 days")
ax.set_xlabel("Period (days)")
ax.set_ylabel("Normalized global wavelet power")
ax.set_title("Global wavelet spectra over the common interval")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Interpretation template

Record the dominant periods returned by the current run, the selected period range, and whether peaks are robust to the COI/record length. Avoid claiming that one series drives the other from these spectra alone. This is a comparison of time-scale structure, not a causal test.
